# Laboratorio 2 - Complejidad y búsqueda de hiperparámetros

En este laboratorio del caso AlpesHearth, se busca fortalecer el modelo inicial de regresión lineal desarrollado previamente para estimar el riesgo cardiovascular, explorando enfoques más avanzados de modelado y evaluando su capacidad de generalización y estabilidad. A partir del conjunto de datos ya depurado, el énfasis estará en la construcción y comparación de modelos de regresión polinomial y regularizada (Ridge y Lasso), utilizando pipelines y validación cruzada con búsqueda sistemática de hiperparámetros. Se analizará el impacto de la complejidad en el desempeño predictivo mediante métricas como RMSE, MAE y R², así como a través de curvas de validación y el estudio del trade-off sesgo-varianza. Posteriormente, se seleccionará el mejor modelo considerando no solo su desempeño promedio sino también su estabilidad, y se estimarán intervalos de confianza mediante bootstrapping para cuantificar la incertidumbre. Finalmente, se realizará un análisis cuantitativo, cualitativo y conceptual de los resultados, comunicando de manera clara las decisiones metodológicas y sus implicaciones técnicas y estratégicas para la organización.

## 1. Importar Librerías

In [76]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import numpy as np
import math

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from statsmodels.formula.api import ols

from statsmodels.stats.diagnostic import linear_rainbow
from scipy.stats import ttest_1samp
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro


## 2. Carga y preparación de los datos

### Carga de los datos:

In [77]:
datos_pacientes = pd.read_csv('./data/Datos Lab 1.csv')
datos2 = datos_pacientes.copy()
datos2.head()

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
0,isDx5313,"November 08, 2023",M,44.0,114.300,1.720,38.600,100.000,112/83,228.0,...,High,N,172.000,0.581,112.0,83.0,Hypertension Stage 1,121.0,19.880,HIGH
1,LHCK2961,20/03/2024,F,57.0,92.923,1.842,33.116,106.315,101/91,158.0,...,High,Y,184.172,0.577,101.0,91.0,Hypertension Stage 2,57.0,16.833,INTERMEDIARY
2,WjVn1699,2021-05-27,F,NaN,73.400,1.650,27.000,78.100,90/74,135.0,...,High,N,165.000,0.473,90.0,74.0,Normal,45.0,12.600,LOW
3,dCDO1109,"April 18, 2022",F,35.0,113.300,1.780,35.800,79.600,92/89,158.0,...,Moderate,Y,178.000,0.447,92.0,89.0,Hypertension Stage 1,94.0,14.920,HIGH
4,pnpE1080,01/11/2024,F,48.0,102.200,1.750,33.400,106.700,121/68,207.0,...,Low,Y,175.000,0.610,121.0,68.0,Elevated,128.0,18.870,HIGH


### Preparación:

In [78]:
datos2 = datos_pacientes.copy()

Como ya se hizo una exploración previa de los datos, ya se puede hacer un segundo modelo de regresión lineal basado en la información que tenemos. Este modelo va a tener aspectos diferentes en términos de preparación de datos con el fin de hacerlo más preciso que el anterior. 

### Completitud

Ya se sabe que la mayoría de las categorías tienen datos faltantes. Esto se comprobó en el modelo pasado usando la siguiente función:

In [79]:
pd.DataFrame({
    "faltantes": datos2.isnull().sum(),
    "porcentaje_%": (datos2.isnull().sum() / len(datos2) * 100).round(2)
})

,faltantes,porcentaje_%
Patient ID,0,0.00
Date of Service,0,0.00
Sex,0,0.00
Age,68,4.15
Weight (kg),73,4.45
Height (m),61,3.72
BMI,53,3.23
Abdominal Circumference (cm),61,3.72
Blood Pressure (mmHg),0,0.00
Total Cholesterol (mg/dL),68,4.15


A diferencia del modelo pasado, en este modelo se van a calcular todas las columnas que pueden ser rellenadas y después sí se van a eliminar las que no pudieron ser calculadas.

#### BMI, Weight (kg), Height (m), Height (cm), Waist-to-Height Ratio, Abdominal Circumference (cm)

In [80]:
# ===== Config: columnas (ajusta si tus nombres difieren) =====
COL_H_M   = "Height (m)"
COL_H_CM  = "Height (cm)"
COL_W     = "Weight (kg)"
COL_BMI   = "BMI"
COL_WHR   = "Waist-to-Height Ratio"
COL_WAIST = "Abdominal Circumference (cm)"

cols_focus = [COL_BMI, COL_H_CM, COL_H_M, COL_W, COL_WHR, COL_WAIST]

# ===== 1) Asegurar tipo numérico en las columnas de interés =====
for c in cols_focus:
    if c in datos2.columns:
        datos2[c] = pd.to_numeric(datos2[c], errors="coerce")

# ===== 2) Definir helpers de "altura válida" para evitar divisiones por cero =====
h_m_valid  = datos2[COL_H_M].notna()  & (datos2[COL_H_M]  > 0)
h_cm_valid = datos2[COL_H_CM].notna() & (datos2[COL_H_CM] > 0)
w_valid    = datos2[COL_W].notna()    & (datos2[COL_W]    > 0)
bmi_valid  = datos2[COL_BMI].notna()  & (datos2[COL_BMI]  > 0)
waist_valid= datos2[COL_WAIST].notna()& (datos2[COL_WAIST]> 0)
whr_valid  = datos2[COL_WHR].notna()  & (datos2[COL_WHR]  > 0)

# ===== 3) Relleno iterativo (porque unas columnas permiten rellenar otras y viceversa) =====
# Repetimos unas cuantas veces hasta estabilizar.
for _ in range(5):
    before = datos2[cols_focus].isna().sum().sum()

    # --- Altura: m <-> cm ---
    # Height (cm) = Height (m) * 100
    mask = datos2[COL_H_CM].isna() & datos2[COL_H_M].notna() & (datos2[COL_H_M] > 0)
    datos2.loc[mask, COL_H_CM] = datos2.loc[mask, COL_H_M] * 100.0

    # Height (m) = Height (cm) / 100
    mask = datos2[COL_H_M].isna() & datos2[COL_H_CM].notna() & (datos2[COL_H_CM] > 0)
    datos2.loc[mask, COL_H_M] = datos2.loc[mask, COL_H_CM] / 100.0

    # Recalcular válidos (pueden haber cambiado)
    h_m_valid  = datos2[COL_H_M].notna()  & (datos2[COL_H_M]  > 0)
    h_cm_valid = datos2[COL_H_CM].notna() & (datos2[COL_H_CM] > 0)

    # --- BMI y Weight (kg) ---
    # BMI = Weight / Height(m)^2
    mask = datos2[COL_BMI].isna() & w_valid & h_m_valid
    datos2.loc[mask, COL_BMI] = datos2.loc[mask, COL_W] / (datos2.loc[mask, COL_H_M] ** 2)

    # Weight = BMI * Height(m)^2
    mask = datos2[COL_W].isna() & bmi_valid & h_m_valid
    datos2.loc[mask, COL_W] = datos2.loc[mask, COL_BMI] * (datos2.loc[mask, COL_H_M] ** 2)

    # Recalcular válidos (pueden haber cambiado)
    w_valid   = datos2[COL_W].notna()   & (datos2[COL_W]   > 0)
    bmi_valid = datos2[COL_BMI].notna() & (datos2[COL_BMI] > 0)

    # --- Waist-to-Height Ratio y Abdominal Circumference ---
    # Asumimos: Waist-to-Height Ratio = Abdominal Circumference (cm) / Height (cm)
    # Ratio = waist_cm / height_cm
    mask = datos2[COL_WHR].isna() & waist_valid & h_cm_valid
    datos2.loc[mask, COL_WHR] = datos2.loc[mask, COL_WAIST] / datos2.loc[mask, COL_H_CM]

    # waist_cm = Ratio * height_cm
    mask = datos2[COL_WAIST].isna() & whr_valid & h_cm_valid
    datos2.loc[mask, COL_WAIST] = datos2.loc[mask, COL_WHR] * datos2.loc[mask, COL_H_CM]

    # Recalcular válidos (pueden haber cambiado)
    waist_valid = datos2[COL_WAIST].notna() & (datos2[COL_WAIST] > 0)
    whr_valid   = datos2[COL_WHR].notna()   & (datos2[COL_WHR]   > 0)

    after = datos2[cols_focus].isna().sum().sum()
    if after == before:  # ya no cambia nada
        break

Se realizaron correcciones de calidad de datos asegurando primero que las variables antropométricas estuvieran en formato numérico válido. Posteriormente, se imputaron valores faltantes utilizando relaciones matemáticas conocidas entre las variables (altura en metros y centímetros, BMI y peso, y la relación cintura-altura), aplicando un proceso iterativo hasta completar la información de forma coherente y consistente sin introducir valores arbitrarios.

#### Systolic + Diastolic <-> Blood Pressure (mmHg)

In [81]:
bp_split = datos2["Blood Pressure (mmHg)"].str.split("/", expand=True)

In [82]:
cond_sys = (
    datos2["Systolic BP"].isna() &
    datos2["Blood Pressure (mmHg)"].notna()
)

datos2.loc[cond_sys, "Systolic BP"] = bp_split.loc[cond_sys, 0].astype(float)

In [83]:
cond_dia = (
    datos2["Diastolic BP"].isna() &
    datos2["Blood Pressure (mmHg)"].notna()
)

datos2.loc[cond_dia, "Diastolic BP"] = bp_split.loc[cond_dia, 1].astype(float)

Se cuentan las líneas faltantes:

In [84]:
pd.DataFrame({
    "faltantes": datos2.isnull().sum(),
    "porcentaje_%": (datos2.isnull().sum() / len(datos2) * 100).round(2)
})


,faltantes,porcentaje_%
Patient ID,0,0.00
Date of Service,0,0.00
Sex,0,0.00
Age,68,4.15
Weight (kg),1,0.06
Height (m),3,0.18
BMI,1,0.06
Abdominal Circumference (cm),1,0.06
Blood Pressure (mmHg),0,0.00
Total Cholesterol (mg/dL),68,4.15


Se puede ver que el número de filas con datos faltantes se redució de manera significativa. Sin embargo, filas que aún tienen datos faltantes que no pueden ser calculados. Para estas filas se puede hacer una imputación con mediana.

Eliminar las que no tienen CVD Risk Score ya que este es el target:

In [85]:
datos2 = datos2.dropna(subset=["CVD Risk Score"])

Eliminar las que faltan para Abdominal Circumference (cm), Waist-to-Height Ratio, Heicht (cm), Height (m), Weight (kg) y BMI ya que el porcentaje de datos faltantes es menor a 1%

In [86]:
datos2 = datos2.dropna(subset=["Waist-to-Height Ratio", "Abdominal Circumference (cm)", "Height (cm)", "Height (cm)", "Weight (kg)", "BMI"])

In [87]:
pd.DataFrame({
    "faltantes": datos2.isnull().sum(),
    "porcentaje_%": (datos2.isnull().sum() / len(datos2) * 100).round(2)
})

,faltantes,porcentaje_%
Patient ID,0,0.00
Date of Service,0,0.00
Sex,0,0.00
Age,67,4.17
Weight (kg),0,0.00
Height (m),0,0.00
BMI,0,0.00
Abdominal Circumference (cm),0,0.00
Blood Pressure (mmHg),0,0.00
Total Cholesterol (mg/dL),68,4.24


Imputación con mediana de Age, Total Cholesterol (mg/dL), HDL (mg/dL), Fasting Blood Sugar (mg/dL), Estimated LDL (mg/dL):

In [88]:
cols_imputar_mediana = [
    "Age",
    "Total Cholesterol (mg/dL)",
    "HDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Estimated LDL (mg/dL)"
]

medianas = {}
for c in cols_imputar_mediana:
    if c in datos2.columns:
        med = datos2[c].median(skipna=True)
        medianas[c] = med
        datos2[c] = datos2[c].fillna(med)

print("Medianas usadas para imputación:")
print(pd.Series(medianas).sort_index())

Medianas usadas para imputación:
Age                             46.0
Estimated LDL (mg/dL)          112.0
Fasting Blood Sugar (mg/dL)    115.0
HDL (mg/dL)                     56.0
Total Cholesterol (mg/dL)      198.0
dtype: float64


In [89]:
pd.DataFrame({
    "faltantes": datos2.isnull().sum(),
    "porcentaje_%": (datos2.isnull().sum() / len(datos2) * 100).round(2)
})

,faltantes,porcentaje_%
Patient ID,0,0.0
Date of Service,0,0.0
Sex,0,0.0
Age,0,0.0
Weight (kg),0,0.0
Height (m),0,0.0
BMI,0,0.0
Abdominal Circumference (cm),0,0.0
Blood Pressure (mmHg),0,0.0
Total Cholesterol (mg/dL),0,0.0


Después de este proceso, no quedan valores faltantes en los datos que serán usados para el modelo. 

### Unicidad

Para los problemas de unicidad, se va a analizar la unicidad por filas y unicidad por Patient ID:

#### Unicidad de filas

Es necesario eliminar estas filas ya que resultados repetidos pueden sesgar el modelo final.

In [90]:
datos2.duplicated().sum()

np.int64(150)

In [91]:
datos2 = datos2.drop_duplicates()

#### Unicidad por Patient ID y Date of Service

Se debe normalizar el formato de la fecha primero:

In [92]:
datos2["Date of Service"] = pd.to_datetime(
    datos2["Date of Service"],
    format="mixed",   # permite múltiples formatos
    dayfirst=True,    # interpreta correctamente DD/MM/YYYY
    errors="coerce"   # valores inválidos → NaT
)

In [93]:
n_dups = datos2.duplicated(subset=["Patient ID", "Date of Service"], keep="first").sum()
print("Filas duplicadas por (Patient ID, Date of Service):", int(n_dups))

Filas duplicadas por (Patient ID, Date of Service): 111


Como estas filas tienen duplicados el Patient ID y Date of Service al mismo tiempo, se van a eliminar, pues no tiene sentido que un paciente tenga datos diferentes en el mismo día. 

In [94]:
datos2 = (
    datos2.sort_values(["Patient ID", "Date of Service"])
      .drop_duplicates(subset=["Patient ID", "Date of Service"], keep="first")
      .reset_index(drop=True)
)

In [95]:
datos2.duplicated(
    subset=["Patient ID", "Date of Service"]
).sum()

np.int64(0)

De esta manera, se eliminaron los duplicados.

### Validez

Además de los rangos específicos, para esta detección de validez, se va a considerar que cualquier dato fuera de Q1-IQR*1.5 o Q3+IQR*1.5 atípico y no se tendrá en cuenta para el modelo.

Los datos faltantes serán imputados con la mediana.

En esta sección se realiza un proceso de validación y corrección de la calidad de los datos numéricos. Primero, se define un conjunto de rangos clínicamente plausibles para cada variable y se identifican valores inválidos, reemplazando números negativos y valores fuera de dichos rangos por valores faltantes (NaN). Posteriormente, se detectan valores atípicos mediante el criterio del rango intercuartílico (IQR × 1.5), los cuales también se consideran inválidos. Finalmente, los valores faltantes generados se imputan utilizando la mediana de cada variable, garantizando un conjunto de datos consistente y completo sin introducir sesgos extremos, y se genera un reporte para verificar que no queden datos faltantes tras la corrección.

In [96]:
df = datos2.copy()


# 1) Define rangos válidos

rangos = {
    "Age": (0, 120),
    "Weight (kg)": (20, 300),
    "Height (m)": (0.9, 2.5),
    "Height (cm)": (90, 250),
    "BMI": (10, 80),
    "Abdominal Circumference (cm)": (40, 200),
    "Waist-to-Height Ratio": (0.2, 1.2),
    "Total Cholesterol (mg/dL)": (50, 500),
    "HDL (mg/dL)": (10, 150),
    "Fasting Blood Sugar (mg/dL)": (40, 400),
    "Systolic BP": (60, 250),
    "Diastolic BP": (30, 150),
    "Estimated LDL (mg/dL)": (0, 400),
    "CVD Risk Score": (0, 100)
}


# 2) Columnas numéricas

num_cols = df.select_dtypes(include="number").columns.tolist()


# 3) Negativos -> NaN

for c in num_cols:
    df.loc[df[c] < 0, c] = np.nan


# 4) Rangos duros -> NaN

for c, (lo, hi) in rangos.items():
    if c in df.columns:
        df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan


# 5) IQR*1.5 -> NaN (solo para columnas numéricas)

for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1

    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    df.loc[(df[c] < low) | (df[c] > high), c] = np.nan


# 6) Imputar con mediana

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())


# 7) Reporte

faltantes_por_col = df[num_cols].isna().sum().sort_values(ascending=False)
print("Faltantes por columna (después de validar + imputar):")
print(faltantes_por_col)

print("\nFilas con al menos 1 faltante en numéricas:", int(df[num_cols].isna().any(axis=1).sum()))

# Guardar resultado
datos2 = df

Faltantes por columna (después de validar + imputar):
Age                             0
Weight (kg)                     0
Height (m)                      0
BMI                             0
Abdominal Circumference (cm)    0
Total Cholesterol (mg/dL)       0
HDL (mg/dL)                     0
Fasting Blood Sugar (mg/dL)     0
Height (cm)                     0
Waist-to-Height Ratio           0
Systolic BP                     0
Diastolic BP                    0
Estimated LDL (mg/dL)           0
CVD Risk Score                  0
dtype: int64

Filas con al menos 1 faltante en numéricas: 0


### Consistencia

Para la revisión y corregimiento de la consistencia, se va a hacer lo siguiente:

- Normalizar y estandarizar variables categóricas
- Convertir Date of Service a fecha
- Recalcular BMI y Waist-to-Height Ratio usando sus componentes
- Asegurarse de que Systolic BP < Diastolic BP
- Recalcula Blood Pressure Category

In [97]:
df = datos2.copy()


# 0) Helpers

def clean_str(s):
    """Limpia strings: strip + upper. Mantiene NaN."""
    return s.astype(str).str.strip().str.upper().replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})

def map_yes_no(series):
    """Convierte múltiples representaciones a 'Y'/'N'."""
    s = clean_str(series)
    mapping = {
        "Y": "Y", "YES": "Y", "SI": "Y", "SÍ": "Y", "TRUE": "Y", "T": "Y", "1": "Y",
        "N": "N", "NO": "N", "FALSE": "N", "F": "N", "0": "N"
    }
    return s.map(mapping).fillna(s)

def bp_category(sys, dia):
    """Categoría de presión arterial (regla típica AHA 2017)."""
    if pd.isna(sys) or pd.isna(dia):
        return np.nan
    # Crisis hipertensiva (opcional): sys>=180 o dia>=120
    # if sys >= 180 or dia >= 120:
    #     return "HYPERTENSIVE CRISIS"
    if sys < 120 and dia < 80:
        return "NORMAL"
    if 120 <= sys < 130 and dia < 80:
        return "ELEVATED"
    if (130 <= sys < 140) or (80 <= dia < 90):
        return "HYPERTENSION STAGE 1"
    if (sys >= 140) or (dia >= 90):
        return "HYPERTENSION STAGE 2"
    return np.nan


# 1) Consistencia de fechas

if "Date of Service" in df.columns:
    df["Date of Service"] = pd.to_datetime(df["Date of Service"], errors="coerce").dt.normalize()


# 2) Consistencia de categóricas

# Sex -> M/F (todo lo demás queda como NaN u OTHER si prefieres)
if "Sex" in df.columns:
    s = clean_str(df["Sex"])
    sex_map = {"M": "M", "MALE": "M", "F": "F", "FEMALE": "F"}
    df["Sex"] = s.map(sex_map)

# Y/N columns
for col in ["Smoking Status", "Diabetes Status", "Family History of CVD"]:
    if col in df.columns:
        df[col] = map_yes_no(df[col])

# Physical Activity Level
if "Physical Activity Level" in df.columns:
    s = clean_str(df["Physical Activity Level"])
    pal_map = {
        "LOW": "LOW", "BAJO": "LOW",
        "MODERATE": "MODERATE", "MODERADO": "MODERATE",
        "HIGH": "HIGH", "ALTO": "HIGH"
    }
    df["Physical Activity Level"] = s.map(pal_map)

# Blood Pressure Category (solo estandariza texto; luego la recalculamos)
if "Blood Pressure Category" in df.columns:
    s = clean_str(df["Blood Pressure Category"])
    bpc_map = {
        "NORMAL": "NORMAL",
        "ELEVATED": "ELEVATED", "ELEVADA": "ELEVATED",
        "HYPERTENSION STAGE 1": "HYPERTENSION STAGE 1",
        "HYPERTENSION STAGE 2": "HYPERTENSION STAGE 2",
        "HIPERTENSIÓN ETAPA 1": "HYPERTENSION STAGE 1",
        "HIPERTENSION ETAPA 1": "HYPERTENSION STAGE 1",
        "HIPERTENSIÓN ETAPA 2": "HYPERTENSION STAGE 2",
        "HIPERTENSION ETAPA 2": "HYPERTENSION STAGE 2",
    }
    df["Blood Pressure Category"] = s.map(bpc_map).fillna(s)


# 3) Coherencia matemática: alturas, BMI, ratio

# Asegurar numéricos relevantes
for c in ["Height (m)", "Height (cm)", "Weight (kg)", "BMI",
          "Abdominal Circumference (cm)", "Waist-to-Height Ratio",
          "Systolic BP", "Diastolic BP"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Altura: recalcular cm y m para que siempre coincidan
h_m = df["Height (m)"] if "Height (m)" in df.columns else None
h_cm = df["Height (cm)"] if "Height (cm)" in df.columns else None

if h_m is not None and h_cm is not None:
    # Si hay Height(m), fuerza Height(cm)
    m_ok = df["Height (m)"].notna() & (df["Height (m)"] > 0)
    df.loc[m_ok, "Height (cm)"] = df.loc[m_ok, "Height (m)"] * 100

    # Si solo hay Height(cm), fuerza Height(m)
    cm_ok = df["Height (cm)"].notna() & (df["Height (cm)"] > 0)
    df.loc[cm_ok, "Height (m)"] = df.loc[cm_ok, "Height (cm)"] / 100

# BMI: recalcular desde Weight y Height(m)
if all(c in df.columns for c in ["Weight (kg)", "Height (m)", "BMI"]):
    w_ok = df["Weight (kg)"].notna() & (df["Weight (kg)"] > 0)
    hm_ok = df["Height (m)"].notna() & (df["Height (m)"] > 0)
    recalc_mask = w_ok & hm_ok
    df.loc[recalc_mask, "BMI"] = df.loc[recalc_mask, "Weight (kg)"] / (df.loc[recalc_mask, "Height (m)"] ** 2)

# Waist-to-Height Ratio: recalcular desde cintura(cm) y altura(cm)
if all(c in df.columns for c in ["Abdominal Circumference (cm)", "Height (cm)", "Waist-to-Height Ratio"]):
    waist_ok = df["Abdominal Circumference (cm)"].notna() & (df["Abdominal Circumference (cm)"] > 0)
    hcm_ok = df["Height (cm)"].notna() & (df["Height (cm)"] > 0)
    recalc_mask = waist_ok & hcm_ok
    df.loc[recalc_mask, "Waist-to-Height Ratio"] = df.loc[recalc_mask, "Abdominal Circumference (cm)"] / df.loc[recalc_mask, "Height (cm)"]


# 4) Consistencia lógica: presión arterial

fixed_swaps = 0
if all(c in df.columns for c in ["Systolic BP", "Diastolic BP"]):
    swap_mask = df["Systolic BP"].notna() & df["Diastolic BP"].notna() & (df["Systolic BP"] < df["Diastolic BP"])
    fixed_swaps = int(swap_mask.sum())
    # swap
    df.loc[swap_mask, ["Systolic BP", "Diastolic BP"]] = df.loc[swap_mask, ["Diastolic BP", "Systolic BP"]].to_numpy()

# Si existe "Blood Pressure (mmHg)" string, puedes opcionalmente reconstruirla desde sys/dia
if all(c in df.columns for c in ["Systolic BP", "Diastolic BP"]) and "Blood Pressure (mmHg)" in df.columns:
    sys_ok = df["Systolic BP"].notna()
    dia_ok = df["Diastolic BP"].notna()
    mask = sys_ok & dia_ok
    df.loc[mask, "Blood Pressure (mmHg)"] = (
        df.loc[mask, "Systolic BP"].round(0).astype(int).astype(str)
        + "/"
        + df.loc[mask, "Diastolic BP"].round(0).astype(int).astype(str)
    )

# Recalcular Blood Pressure Category desde sys/dia
recalc_cat_count = 0
if all(c in df.columns for c in ["Systolic BP", "Diastolic BP", "Blood Pressure Category"]):
    before = df["Blood Pressure Category"].copy()
    df["Blood Pressure Category"] = df.apply(lambda r: bp_category(r["Systolic BP"], r["Diastolic BP"]), axis=1)
    recalc_cat_count = int((before != df["Blood Pressure Category"]).sum(skipna=False))


# 5) Reporte simple

print("=== REPORTE DE CONSISTENCIA ===")
print("Swaps hechos (Systolic < Diastolic):", fixed_swaps)
if "Blood Pressure Category" in df.columns:
    print("Cambios en Blood Pressure Category (aprox):", recalc_cat_count)

# Valores únicos (para revisar rápido)
for col in ["Sex", "Smoking Status", "Diabetes Status", "Family History of CVD", "Physical Activity Level", "Blood Pressure Category"]:
    if col in df.columns:
        print(f"\nValores únicos en {col}:")
        print(df[col].value_counts(dropna=False))

# Guardar resultado
datos2 = df


=== REPORTE DE CONSISTENCIA ===
Swaps hechos (Systolic < Diastolic): 52
Cambios en Blood Pressure Category (aprox): 11

Valores únicos en Sex:
Sex
F    676
M    668
Name: count, dtype: int64

Valores únicos en Smoking Status:
Smoking Status
Y    691
N    653
Name: count, dtype: int64

Valores únicos en Diabetes Status:
Diabetes Status
Y    679
N    665
Name: count, dtype: int64

Valores únicos en Family History of CVD:
Family History of CVD
N    685
Y    659
Name: count, dtype: int64

Valores únicos en Physical Activity Level:
Physical Activity Level
HIGH        462
MODERATE    449
LOW         433
Name: count, dtype: int64

Valores únicos en Blood Pressure Category:
Blood Pressure Category
HYPERTENSION STAGE 2    552
HYPERTENSION STAGE 1    445
NORMAL                  258
ELEVATED                 89
Name: count, dtype: int64


Se crea una copia del dataset y se define la variable objetivo (CVD Risk Score) junto con las variables predictoras seleccionadas. Posteriormente, se separan las características (X) y la variable a predecir (y).

In [98]:
data = datos2.copy()

target = "CVD Risk Score"

feature_cols = [
    "Age",
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Family History of CVD",
    "Physical Activity Level",
    "BMI",
    "Abdominal Circumference (cm)",
    "Systolic BP",
    "Diastolic BP",
    "Estimated LDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
]

X = data[feature_cols].copy()
y = data[target].copy()


El conjunto de datos se divide en entrenamiento (75%) y prueba (25%) para evaluar el modelo de forma objetiva. Además, se muestran las dimensiones resultantes de cada subconjunto.

In [99]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1008, 12), (336, 12), (1008,), (336,))

Se define una función de limpieza que convierte variables binarias a formato numérico (1/0) y estandariza valores categóricos como sexo y nivel de actividad física para asegurar consistencia en los datos. Esta función se integra en un FunctionTransformer para usarla dentro de un pipeline.

In [100]:
def limpiar_columnas(df):
    df = df.copy()

    # Asegurar Y/N -> 1/0 (por si aún quedara en texto)
    yn_cols = ["Smoking Status", "Diabetes Status", "Family History of CVD"]
    mapping = {"Y": 1, "N": 0, "YES": 1, "NO": 0, "TRUE": 1, "FALSE": 0, "1": 1, "0": 0}

    for col in yn_cols:
        if col in df.columns:
            s = df[col]

            # Si viene como texto, mapeamos. Si viene numérico, lo dejamos como número.
            if pd.api.types.is_string_dtype(s) or s.dtype == "object":
                s = s.astype(str).str.strip().str.upper()
                df[col] = s.map(mapping).astype(float)
            else:
                df[col] = pd.to_numeric(s, errors="coerce")

    # Estandarizar Sex y Physical Activity Level (por si hay variaciones)
    if "Sex" in df.columns:
        df["Sex"] = df["Sex"].astype(str).str.strip().str.upper().replace({"MALE":"M", "FEMALE":"F", "NAN": np.nan})

    if "Physical Activity Level" in df.columns:
        df["Physical Activity Level"] = (
            df["Physical Activity Level"].astype(str).str.strip().str.upper()
            .replace({"MODERADO":"MODERATE", "BAJO":"LOW", "ALTO":"HIGH", "NAN": np.nan})
        )

    return df

limpieza = FunctionTransformer(limpiar_columnas)

Se vuelve a definir la misma función de limpieza y se encapsula nuevamente como transformador, permitiendo aplicar automáticamente la estandarización y codificación de variables durante el flujo de preprocesamiento del modelo.

In [101]:
numeric_features = [
    "Age",
    "BMI",
    "Abdominal Circumference (cm)",
    "Systolic BP",
    "Diastolic BP",
    "Estimated LDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Smoking Status",
    "Diabetes Status",
    "Family History of CVD",
]

categorical_features = ["Sex", "Physical Activity Level"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline_regresion = Pipeline(steps=[
    ("limpieza", limpieza),
    ("preprocesamiento", preprocessor),
])

Se aplica el pipeline de preprocesamiento al conjunto de entrenamiento y se transforman las variables para dejarlas listas para el modelo. Luego, se entrena un modelo de regresión lineal utilizando las características transformadas.

In [102]:
Xt_train = pipeline_regresion.fit_transform(X_train)

feature_names = pipeline_regresion.named_steps["preprocesamiento"].get_feature_names_out()

Xt_train_df = pd.DataFrame(
    Xt_train.toarray() if hasattr(Xt_train, "toarray") else Xt_train,
    columns=feature_names,
    index=X_train.index
)

Modelo = LinearRegression()
Modelo.fit(Xt_train_df, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


### Validación del modelo

Se generan predicciones sobre el conjunto de entrenamiento y se calculan las métricas MAE, RMSE y R² para evaluar el desempeño del modelo durante la fase de entrenamiento.

In [103]:
y_train_pred = Modelo.predict(Xt_train_df)

mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = r2_score(y_train, y_train_pred)

mae_train_m2 = mae_train
rmse_train_m2 = rmse_train
r2_train_m2 = r2_train

print("MAE  train:", mae_train)
print("RMSE train:", rmse_train)
print("R²   train:", r2_train)

MAE  train: 0.8998411971290402
RMSE train: 1.2259286182481122
R²   train: 0.7533479868628499


## 4. Construcción del Pipeline de Preprocesamiento

### 4.1 Selección y exclusión de variables

### 4.2 Separación por tipo de variable (numéricas y categóricas)

### 4.3 Transformaciones de calidad de los datos

### 4.4 Generación de características polinomiales

### 4.5 Ensamblaje del pipeline completo

## 5. Construcción del modelo de regresión polinomial

### 5.1 Definición del modelo base

### 5.2 Definición del espacio de hiperparámetros

### 5.3 Búsqueda de hiperparámetros con GridSearchCV

## 6. Evaluación del modelo polinomial

### 6.1 Mejor configuración encontrada

### 6.2 Predicciones sobre el conjunto de prueba

### 6.3 Reporte de métricas

## 7. Curvas de validación y análisis de complejidad

### 7.1 Generación de validation curves

### 7.2 Visualización del desempeño

### 7.3 Análisis sesgo-varianza

## 8. Modelo Ridge (Regularización L2)

### 8.1 Construcción del pipeline Ridge

### 8.2 Definición de hiperparámetros (α)

### 8.3 Búsqueda con GridSearchCV

### 8.4 Evaluación del modeo Ridge (Métricas)

## 9. Modelo Lasso (Regularización L1)

### 9.1 Construcción del pipeline Ridge

### 9.2 Definición de hiperparámetros (α)

### 9.3 Búsqueda con GridSearchCV

### 9.4 Evaluación del modeo Ridge (Métricas)

### 9.5 Análisis de Coeficientes

## 10. Construcción del modelo polinomial regularizado

### 10.1 Pipeline combinado

### 10.2 Espacio de hiperparámetros

### 10.3 Búsqueda con GridSearchCV

## 11. Evaluación del modelo polinomial regularizado

### 11.1 Mejor configuración encontrada

### 11.2 Evaluación en test (Métricas)

## 12. Comparación global de modelos

### 12.1 Tabla comparativa de resultados

### 12.2 Análisis de estabilidad vs desempeño

### 12.3 Selección del mejor modelo

### 12.4 Justificación metodológica de la elección

## 13. Construcción de intervalos de confianza (Bootstrapping)

### 13.1 Definición del procedimiento de remuestreo

### 13.2 Ejecución de boostrapping

### 13.3 Cálculo de intervalos de confianza

### 13.4 Visualización de distribuciones de error

### 13.5 Análisis de estabilidad y confiabilidad

## 14. Análisis cuantitativo

¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?

¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide, ¿cuál puede ser la explicación?

¿El modelo con mejor métrica promedio es necesariamente el más adecuado? Justifica considerando también la desviación estándar del desempeño.

Con base en las curvas de validación, ¿cómo cambia el error a medida que aumenta la complejidad? ¿En qué punto se evidencia sobreajuste?

¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?

¿Los intervalos de confianza obtenidos mediante bootstrapping sugieren estabilidad o alta variabilidad en el desempeño? ¿Qué implicaciones tiene esto?

## 15. Análisis cualitativo

¿Qué relación observas entre complejidad del modelo, capacidad de generalización y estabilidad del desempeño?

¿Qué fuentes de sesgo podrían estar presentes en los datos o en el proceso de modelado?

Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad de los modelos? Explique.